# Top-K Document Recommender with Cosine Similarity

**Activity 3 | Knowledge Management Systems and Technologies** | Path A: TF-IDF

## Introduction
In an organization, useful knowledge is usually spread across policies, incident tickets, FAQs, and meeting minutes, and employees spend time looking for a document that was already written or solving a problem that was already solved. A recommender system addresses this situation by suggesting related documents from a seed document or from a short text.

This notebook implements a Top-K recommender built with TF-IDF and cosine similarity. It loads a simulated corpus of 31 documents, cleans the text, builds the document vectors, and returns the five most similar documents with their similarity scores. The notebook also includes three validation tests (Policy/Process, Support/Incidents, and free text) and the tests of input validation.

**Dependencies:** `pandas` and `scikit-learn` (see `requirements.txt`). The notebook must be executed in the same folder as `corpus.csv`.

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 80)

## Corpus

The corpus (`corpus.csv`) contains the fields `id`, `title`, `text`, and `category`. The function `load_corpus` validates that the file exists, that the required columns are present, and that the identifiers are unique, since a duplicated ID would make a seed document ambiguous.

In [2]:
CORPUS_PATH = Path("corpus.csv")
REQUIRED_COLUMNS = {"id", "title", "text"}


def load_corpus(path: Path = CORPUS_PATH) -> pd.DataFrame:
    """Load the corpus and validate that the file exists, has the required columns, and has unique IDs."""
    if not Path(path).exists():
        raise FileNotFoundError(f"The corpus file was not found: {path}")
    df = pd.read_csv(path, dtype=str).fillna("")
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"The corpus is missing the required columns: {sorted(missing)}")
    if df["id"].duplicated().any():
        raise ValueError("The corpus contains duplicated identifiers.")
    return df


corpus = load_corpus()
print(f"Documents loaded: {len(corpus)}")
print(f"Unique IDs: {corpus['id'].is_unique}")
corpus["category"].value_counts().rename_axis("category").to_frame("documents")

Documents loaded: 31
Unique IDs: True


,documents
category,
Policy/Process,11
Support/Incidents,10
FAQ,5
Meeting,5


In [3]:
corpus[["id", "title", "category"]].head(10)

,id,title,category
0,DOC001,Corporate Password Policy,Policy/Process
1,DOC002,New Employee Onboarding Process,Policy/Process
2,DOC003,Employee Offboarding and Access Revocation Process,Policy/Process
3,DOC004,Acceptable Use Policy for Equipment and Software,Policy/Process
4,DOC005,Vacation Request Procedure,Policy/Process
5,DOC006,Travel Expense Reimbursement Policy,Policy/Process
6,DOC007,System Change Management Process,Policy/Process
7,DOC008,Backup and Data Retention Policy,Policy/Process
8,DOC009,Remote Work and VPN Access Policy,Policy/Process
9,DOC010,Computer Equipment Procurement Process,Policy/Process


## Text Cleaning

Minimal cleaning is applied to every document: lowercase conversion, accent removal, and removal of special characters. English stopwords are also removed, since they carry little meaning and would add noise to the scores. Finally, a light stemming (the first six letters of each word) is applied so that variants such as “password” and “passwords” count as the same term.

In [4]:
STOPWORDS_EN = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "can", "could", "do", "does",
    "for", "from", "had", "has", "have", "i", "if", "in", "into", "is", "it", "its", "may", "must",
    "my", "no", "not", "of", "on", "or", "our", "should", "so", "than", "that", "the", "their",
    "them", "then", "there", "these", "they", "this", "to", "was", "we", "were", "what", "when",
    "which", "who", "will", "with", "within", "you", "your", "because", "after", "all", "any",
    "each", "every", "only", "also", "such", "up", "out", "how", "cannot", "while", "other",
}


def clean_text(text: str) -> str:
    """Apply minimal cleaning: lowercase, accent removal, and special-character removal."""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def remove_stopwords(text: str) -> str:
    """Remove the words listed in STOPWORDS_EN from an already cleaned text."""
    return " ".join(w for w in text.split() if w not in STOPWORDS_EN)


def stem(word: str) -> str:
    """Truncate a word to six letters, so that variants such as password/passwords count as one term."""
    return word[:6]


def tokenize(text: str):
    """Split a cleaned text into stemmed tokens; words of one or two letters are ignored."""
    return [stem(w) for w in text.split() if len(w) > 2]


example = "Passwords are renewed every 90 days, and sharing credentials by email is PROHIBITED!"
cleaned = remove_stopwords(clean_text(example))
print("Original :", example)
print("Cleaned  :", cleaned)
print("Tokens   :", tokenize(cleaned))

Original : Passwords are renewed every 90 days, and sharing credentials by email is PROHIBITED!
Cleaned  : passwords renewed 90 days sharing credentials email prohibited
Tokens   : ['passwo', 'renewe', 'days', 'sharin', 'creden', 'email', 'prohib']


## Recommender

The class `Recommender` builds the TF-IDF matrix once and answers two types of requests: by document ID (`by_id`) and by free text (`by_query`). The title is concatenated with the text because it adds topical signal. When an ID is used, the seed document is excluded from the results, since a document must not recommend itself.

In [5]:
class Recommender:
    """Build the TF-IDF matrix once and answer queries by document ID or by free text."""

    def __init__(self, corpus: pd.DataFrame):
        self.df = corpus.reset_index(drop=True)
        self.df["clean"] = (self.df["title"] + " " + self.df["text"]).map(
            lambda t: remove_stopwords(clean_text(t))
        )
        self.vectorizer = TfidfVectorizer(
            tokenizer=tokenize, token_pattern=None, lowercase=False, sublinear_tf=True
        )
        self.matrix = self.vectorizer.fit_transform(self.df["clean"])
        self.id_to_idx = {doc_id: i for i, doc_id in enumerate(self.df["id"])}

    def _top_k(self, scores, k, exclude_idx=None):
        """Return the k documents with the highest scores as (id, title, category, score) tuples."""
        results = []
        for idx in scores.argsort()[::-1]:
            if idx == exclude_idx:
                continue  # the seed document must not recommend itself
            results.append((self.df.loc[idx, "id"], self.df.loc[idx, "title"],
                            self.df.loc[idx].get("category", ""), float(scores[idx])))
            if len(results) == k:
                break
        return results

    def by_id(self, doc_id: str, k: int = 5):
        """Recommend documents similar to the seed document identified by doc_id."""
        doc_id = doc_id.strip().upper()
        if doc_id not in self.id_to_idx:
            raise KeyError(f"The identifier '{doc_id}' does not exist in the corpus. A valid example is {self.df['id'].iloc[0]}.")
        i = self.id_to_idx[doc_id]
        scores = cosine_similarity(self.matrix[i], self.matrix).ravel()
        return self._top_k(scores, k, exclude_idx=i)

    def by_query(self, query: str, k: int = 5):
        """Recommend documents similar to a free-text query."""
        cleaned = remove_stopwords(clean_text(query or ""))
        if not cleaned:
            raise ValueError("The query is empty.")
        vec = self.vectorizer.transform([cleaned])
        if vec.nnz == 0:
            raise ValueError("None of the query terms appear in the vocabulary of the corpus.")
        scores = cosine_similarity(vec, self.matrix).ravel()
        return self._top_k(scores, k)


rec = Recommender(corpus)
print(f"TF-IDF matrix: {rec.matrix.shape[0]} documents x {rec.matrix.shape[1]} terms")

TF-IDF matrix: 31 documents x 394 terms


## Execution Interface

The function `recommend` is the interface of the notebook. It accepts a document ID, a free-text query, or both, and prints the Top-K table with the similarity scores. Errors are shown as a message instead of stopping the notebook, which allows the validation tests to be executed in sequence.

In [6]:
def show(results, header):
    """Print the ranked results as a table with their similarity scores."""
    print("=" * 80)
    print(header)
    print("-" * 80)
    print(f"{'#':<3}{'ID':<8}{'Score':<9}{'Category':<19}Title")
    for n, (doc_id, title, cat, score) in enumerate(results, 1):
        print(f"{n:<3}{doc_id:<8}{score:<9.4f}{cat:<19}{title}")
    print("=" * 80)


def recommend(seed_id: str = None, query: str = None, k: int = 5):
    """Print the Top-K recommendations for a seed document ID, a free-text query, or both."""
    if seed_id is None and query is None:
        print("Error: provide a document ID, a query, or both.")
        return
    try:
        if seed_id is not None:
            row = rec.df[rec.df["id"] == seed_id.strip().upper()]
            label = f"{seed_id.upper()} - {row['title'].iloc[0]}" if len(row) else seed_id
            print(f"INPUT (ID): {label}")
            show(rec.by_id(seed_id, k), f"Top-{k} recommendations")
        if query is not None:
            print(f'INPUT (text): "{query}"')
            show(rec.by_query(query, k), f"Top-{k} recommendations")
    except (KeyError, ValueError) as e:
        print(f"Error: {e}")

## Validation Tests

Three distinct tests are executed: a seed document from Policy/Process, a seed document from Support/Incidents, and a free-text query. Each test is followed by a brief comment on the coherence of the recommendations.

### Test 1: Seed document from Policy/Process (DOC001)

In [7]:
recommend(seed_id="DOC001")

INPUT (ID): DOC001 - Corporate Password Policy
Top-5 recommendations
--------------------------------------------------------------------------------
#  ID      Score    Category           Title
1  DOC022  0.1930   FAQ                FAQ: How do I reset my password?
2  DOC016  0.1876   Support/Incidents  Incident: Phishing Attempt Reported by Employees
3  DOC025  0.1519   FAQ                FAQ: What should I do if I receive a suspicious email?
4  DOC005  0.1453   Policy/Process     Vacation Request Procedure
5  DOC027  0.1348   Meeting            Minutes: Quarterly Information Security Review


**Comment.** The seed belongs to Policy/Process (password policy). The top results are coherent: DOC022 (FAQ on resetting a password) shares the terms “password”, “reset”, and “policy”; DOC016 (phishing incident) and DOC025 (FAQ on suspicious emails) deal with credential theft, a closely related security topic; and DOC027 (security review minutes) mentions updating the password policy. The weakest result is DOC005 (vacation request procedure, rank 4), which matches only generic process words such as “days” and “employees”. This shows a limitation of keyword-based similarity.

### Test 2: Seed document from Support/Incidents (DOC014)

In [8]:
recommend(seed_id="DOC014")

INPUT (ID): DOC014 - Incident: VPN Connection Failure for Remote Workers
Top-5 recommendations
--------------------------------------------------------------------------------
#  ID      Score    Category           Title
1  DOC009  0.3236   Policy/Process     Remote Work and VPN Access Policy
2  DOC026  0.1445   FAQ                FAQ: How do I connect to the network from home?
3  DOC013  0.1214   Support/Incidents  Ticket: User Locked Out Due to Expired Password
4  DOC020  0.1171   Support/Incidents  Incident: Network Connectivity Loss at the North Branch
5  DOC016  0.0964   Support/Incidents  Incident: Phishing Attempt Reported by Employees


**Comment.** The seed belongs to Support/Incidents (VPN failure for remote workers). The first two results are very relevant: DOC009 (remote work and VPN policy, 0.32) and DOC026 (FAQ on connecting from home) share the terms VPN, remote, and connect, so the analyst receives both the governing policy and the end-user guide. DOC013 (locked account) and DOC020 (network connectivity loss) are related access incidents, which makes them reasonable similar cases. DOC016 (phishing, 0.10) is the weakest result and appears only through generic terms such as “users” and “access”.

### Test 3: Free-text query

In [9]:
recommend(query="I can't log in because I forgot my password")

INPUT (text): "I can't log in because I forgot my password"
Top-5 recommendations
--------------------------------------------------------------------------------
#  ID      Score    Category           Title
1  DOC022  0.3048   FAQ                FAQ: How do I reset my password?
2  DOC013  0.2327   Support/Incidents  Ticket: User Locked Out Due to Expired Password
3  DOC001  0.1491   Policy/Process     Corporate Password Policy
4  DOC026  0.1122   FAQ                FAQ: How do I connect to the network from home?
5  DOC016  0.0659   Support/Incidents  Incident: Phishing Attempt Reported by Employees


**Comment.** The input is a free-text query. DOC022 (FAQ: how do I reset my password, 0.30) ranks first, followed by DOC013 (expired password ticket, 0.23) and DOC001 (password policy). This combination of self-service guide, solved case, and policy is coherent and useful. Ranks 4 and 5 (DOC026 and DOC016) are weak, with scores below 0.12, because “log in” has no semantic link with “connect” or “access” in TF-IDF. Synonyms such as “sign in” would not be recognized.

## Input Validation

The interface handles an invalid identifier, an empty query, and a query whose terms do not appear in the corpus. A missing corpus file is handled by `load_corpus`.

In [10]:
recommend(seed_id="DOC999")        # invalid identifier
recommend(query="")                # empty query
recommend(query="zzzz qqqq")       # no known terms
try:
    load_corpus(Path("nope.csv"))  # missing file
except FileNotFoundError as e:
    print(f"Error: {e}")

INPUT (ID): DOC999
Error: "The identifier 'DOC999' does not exist in the corpus. A valid example is DOC001."
INPUT (text): ""
Error: The query is empty.
INPUT (text): "zzzz qqqq"
Error: None of the query terms appear in the vocabulary of the corpus.
Error: The corpus file was not found: nope.csv


## Optional: Interactive Mode

The following cell allows the user to type a document ID or a free text. It is commented out so that the notebook can be executed from beginning to end without waiting for input. To use it, remove the comment marks and execute the cell.

In [11]:
# while True:
#     entry = input("Type a document ID (e.g. DOC001), free text, or 'quit': ").strip()
#     if entry.lower() in {"quit", "exit", "q"}:
#         break
#     if re.fullmatch(r"DOC\d+", entry.upper()):
#         recommend(seed_id=entry)
#     else:
#         recommend(query=entry)

## Conclusion

The recommender fulfills its purpose, since it returns five ranked documents with similarity scores for both a document ID and a free-text query, and its top results were coherent in the three tests: for example, the VPN incident led to the remote work policy and the connection FAQ. Its main strength is transparency, because each score can be traced to shared words. Its main weakness is its dependence on exact vocabulary, so synonyms and generic words reduce the quality of the lower-ranked results.

For these reasons, the approach is suitable as a first version that reduces rework and accelerates decision-making, while a real environment would require embeddings, metadata, role-based filters, security controls, and continuous evaluation, as described in the report (`report.pdf`).